# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 19 — The paper's own standard, applied to the results that favour it

Three of this paper's results sit on a looser footing than the results it uses to criticise
others, and the asymmetry runs the way the paper says it usually runs.

The regime decomposition is called the most informative result of the attribution analysis and
appears in the abstract. It is measured on the lift scale, which the paper demotes in favour
of a threshold-free measure two sections earlier. The panel of eleven structural rankings
selected the baseline on one partition and was never repeated on the corrected ones, though
the baseline's own agreement varies across them by more than any effect reported here. And the
specificity check, whose conclusion also reaches the abstract, comes from the partition drawn
with the splitter this paper discredits.

A fourth claim is not about attribution at all. The accuracy section says the non-nested
hyperparameter search presumably affects the models alike, and says it was not tested. The
out-of-fold predictions are stored, so it can be tested without fitting anything.

None of the four needs new models. This notebook computes all four.

**Contents**

1. Environment, data and stored quantities
2. The regime decomposition on the threshold-free measure
3. The structural ranking panel on the corrected partitions
4. The specificity check on the corrected partitions
5. What the non-nested search cost
6. Summary

*Approximate runtime: about an hour and a half on a GPU. Section 4 recomputes signed
attributions for twenty-five models -- ten on the designated partition as a control and five
on each of three others -- over two evaluation sets each, and is the only part that needs a
GPU; Sections 2, 3 and 5 are numpy, and Section 5 takes a
couple of minutes.*

## Pre-registered readings

Fixed before the notebook runs.

### The regime decomposition on the threshold-free measure

| result | reading |
|---|---|
| both signs match the lift result and both intervals exclude zero | the decomposition is reported on the threshold-free measure as primary, and the lift version becomes the sensitivity analysis |
| signs match, at least one interval contains zero | the direction holds on both measures and the size is resolved on one; both are reported and the abstract quotes the threshold-free difference with its interval |
| either sign flips | the decomposition is a property of the lift measure rather than of the models, the abstract changes, and the finding is reported as measure-dependent |

The tie fraction is computed per regime at the same time, under two definitions: the share of
the retained set decided at the cut, which is what the manuscript reports, and the share that
a tie makes genuinely arbitrary, which excludes the case where a tied block fits the retained
set exactly. If the two regimes differ sharply, the lift measure was doing different work in
each, and that is reported whichever way it falls.

**On units.** AUROC and lift are not the same scale, and AUROC is bounded above by one while
lift is not. If the first branch holds, the abstract will quote the AUROC differences with
their intervals and, beside them, each as a percentage of the attainable range for that
regime; it will not quote a lift difference and an AUROC difference as though they were
comparable. The published lift figures move to the sensitivity analysis with their own
units named. Fixing this now prevents an argument about a shrinking number after the fact.

### The structural ranking panel on partitions 201, 202 and 203

| result | reading |
|---|---|
| atom degree lies above chance under all three regimes on all three partitions, and is the only ranking that does | the selection rule transports, and the baseline choice is reported as validated out of sample |
| degree passes but another ranking passes too | the rule does not single out one winner out of sample. The manuscript says so, and states that the choice then rests on the secondary criteria already given in the Methods: degree is the simplest of them, needs no parameter, and had a mechanism available in advance |
| it fails on one partition | the rule is reported as partition-dependent, and the manuscript says which partition and which regime |
| another ranking beats degree on the same rule in two or more partitions | the baseline choice was an artefact of the designated partition; the manuscript reports the alternative and reconsiders the recommendation |

### The specificity check on the corrected partitions

| result | reading |
|---|---|
| GIN's difference of differences is negative on all three **and its mutagenic component is positive on the same partition** | the abstract claim stands and is reported as replicated |
| negative but with a mutagenic component below zero | the number replicates and the reading does not: a model below the baseline on both sets, further below on the non-mutagenic one, produces the same sign for the opposite reason. Reported as such |
| negative on two of three | reported as replicating in direction with one exception named |
| positive on any partition | the claim is withdrawn from the abstract and reported as specific to the designated partition |

Five models per partition against ten on the designated one, so intervals will be wider here
whatever the effect. A wider interval is not a failure to replicate and will not be read as
one.

**On the controls.** Each of Sections 2, 3 and 4 reproduces a published quantity before it
computes a new one, and every function in this notebook is a rewrite. A control that misses
its published value by more than the stated tolerance halts the section. If that happens the
tolerance is not widened: the discrepancy is traced to the code path that produced it, and if
it cannot be traced it is reported. The tolerances are 0.005 on an AUROC or a difference of
differences, 0.01 on a lift, and 0.002 on an out-of-fold ROC-AUC published to three
decimals. They are fixed here and not after seeing a number.

### What the non-nested search cost

The optimism of a model on the 725 compounds the hyperparameter search saw, against the 6,530
it did not. The absolute gap is expected and is not the question; the two subsets are not
matched on difficulty and only the spread across models is interpretable.

| result | reading |
|---|---|
| the spread of optimism across the six models is small relative to the $+0.0146$ family contrast | the contrast is not attributable to tuning exposure, and the manuscript replaces its assumption with this measurement |
| the spread is of the same order as the contrast, or the two families differ systematically | the contrast cannot be separated from tuning exposure in this design, and the accuracy section says so |
| the graph and classical means differ in the direction that would inflate the contrast | the reported contrast is an upper bound and the manuscript states by how much |

## 1. Environment, data and stored quantities

Part 17 installed without a pin and its versions were not recorded, so they cannot be matched
here. Canonical SMILES, Murcko scaffolds and state-dict key names are all version sensitive
and this notebook reloads weights fitted in Part 17, so the versions in use are printed and
recorded instead. What guards the reconstruction is not a pin but the anchors each section
carries: the compound count, the evaluation set sizes, and the published values every section
reproduces before it computes a new one.

In [1]:
!pip install rdkit
!pip install torch_geometric

import rdkit, torch_geometric, sys
print(f"python {sys.version.split()[0]}  rdkit {rdkit.__version__}  "
      f"torch_geometric {torch_geometric.__version__}")
# Installed without -q on purpose: a failed resolution here is the cause of every later
# import error, and suppressing it moves the symptom two cells away from the fault.
# Record the line above with the results; Part 17 did not record its own.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.4 MB/s eta 0:00:00
python 3.13.15  rdkit 2026.03.5  torch_geometric 2.8.0.post1


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = "/content/drive/MyDrive/ames_pipeline"
DATADIR, R, EXPL, MODELS = f"{WORK}/data", f"{WORK}/results", f"{WORK}/explain", f"{WORK}/models"
os.makedirs(R, exist_ok=True)
print("results:", R)

Mounted at /content/drive
results: /content/drive/MyDrive/ames_pipeline/results


In [3]:
%%writefile /content/ames9.py
"""Definitions shared across Part 9, matching the protocol used in Parts 2 to 5."""
import random, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator as rfg
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*"); warnings.filterwarnings("ignore")
FP_RADIUS, FP_BITS = 2, 2048
ATOM_DIM, BOND_DIM = 24, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']
morgan_gen = rfg.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return ""
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try: return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception: return smi

def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs = list(groups.values()); rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr)+len(g) <= a: tr += g
        elif len(tr)+len(va)+len(g) <= b: va += g
        else: te += g
    return d.loc[tr], d.loc[va], d.loc[te]

def fp_array(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(morgan_gen.GetFingerprint(mol), arr)
    return arr

def build_X(d):
    X, keep = [], []
    for i, row in d.iterrows():
        a = fp_array(row["canonical"])
        if a is not None: X.append(a); keep.append(i)
    return np.array(X), d.loc[keep].reset_index(drop=True)

def one_hot(x, allowable):
    if x not in allowable: x = allowable[-1]
    return [int(x == a) for a in allowable]

def atom_features(a):
    f  = one_hot(a.GetSymbol(), ATOM_LIST) + one_hot(str(a.GetHybridization()), HYBRIDIZATION)
    f += [a.GetDegree(), a.GetFormalCharge(), int(a.GetIsAromatic()),
          a.GetTotalNumHs(), int(a.IsInRing())]
    return f

def bond_features(b):
    bt = b.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(b.GetIsConjugated()), int(b.IsInRing())]

def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bf = bond_features(b)
        ei += [[i, j], [j, i]]; ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)

def build_dataset(d):
    g = [smiles_to_data(r["canonical"], float(r["label"])) for _, r in d.iterrows()]
    return [x for x in g if x is not None]

class GNN(nn.Module):
    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn": conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else: raise ValueError(kind)
            self.convs.append(conv); self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, data):
        x, ei, ea, b = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(x, ei, ea) if self.use_edge_attr else conv(x, ei)
            h = F.relu(bn(h)); h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, b), global_add_pool(x, b)], dim=1)
        return self.head(g).squeeze(-1)

def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4, use_edge_attr=True):
    return GNN(kind, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads, use_edge_attr=use_edge_attr).to(DEVICE)

class TensorView:
    def __init__(self, x, edge_index, edge_attr, batch):
        self.x, self.edge_index, self.edge_attr, self.batch = x, edge_index, edge_attr, batch

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps = [], []
    for d in loader:
        d = d.to(DEVICE)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(model(d)).cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

def train_gnn(kind, params, seed, tr_set, va_set, pos_weight,
              max_epochs=250, patience=30, use_edge_attr=True, epoch_hook=None):
    """The protocol used for every model in this study."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr)
    opt = torch.optim.Adam(model.parameters(), lr=params["lr"],
                           weight_decay=params["weight_decay"])
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    tl = DataLoader(tr_set, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    vl = DataLoader(va_set, batch_size=256)
    best, best_state, waited = -1.0, None, 0
    for ep in range(max_epochs):
        model.train()
        for d in tl:
            d = d.to(DEVICE); opt.zero_grad()
            crit(model(d), d.y.view(-1)).backward(); opt.step()
        y, p = evaluate(model, vl)
        auc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else 0.5
        if epoch_hook is not None: epoch_hook(ep, model, auc)
        if auc > best:
            best, waited = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience: break
    model.load_state_dict(best_state); model.eval()
    return model, best

def train_to_fit(kind, params, seed, dataset, n_epochs, dropout=0.0, weight_decay=0.0,
                 lr=None):
    """Fit a target with regularisation and early stopping switched off."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], dropout,
                      params.get("residual", True))
    opt = torch.optim.Adam(model.parameters(), lr=lr or params["lr"],
                           weight_decay=weight_decay)
    crit = nn.BCEWithLogitsLoss()
    ld = DataLoader(dataset, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    last = float("nan")
    for ep in range(n_epochs):
        model.train(); tot = 0.0
        for d in ld:
            d = d.to(DEVICE); opt.zero_grad()
            loss = crit(model(d), d.y.view(-1)); loss.backward(); opt.step()
            tot += loss.item()
        last = tot / max(len(ld), 1)
    model.eval(); return model, last

Writing /content/ames9.py


In [4]:
import sys, json, glob, re
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, torch
from ames9 import *
from scipy import stats
from sklearn.metrics import roc_auc_score as _auc
from collections import defaultdict, Counter

SEED, N_BOOT, K_FRAC, N_TIE = 42, 10000, 0.30, 200
PART_SEEDS = [201, 202, 203]
HP_ALL = json.load(open(f"{R}/hpo_gnn.json"))

def cached(name, fn):
    p = f"{R}/{name}"
    if os.path.exists(p):
        print(f"  {name}: loaded from disk"); return pd.read_csv(p)
    out = fn()
    # a note column is normal; a non-empty note marks a stage that could not run. Testing
    # for the column rather than its contents would refuse to cache every table that has one.
    skipped = ("note" in out.columns
               and out["note"].fillna("").astype(str).str.len().gt(0).any())
    if skipped or not len(out):
        print(f"  {name}: stage could not run, not cached"); return out
    out.to_csv(p, index=False); print(f"  {name}: written"); return out

def rows_ok(frame):
    """The rows a stage actually computed.

    A note column whose entries are all the empty string round-trips through CSV as all-NaN
    floats, so a cached table tested with note == "" yields nothing and the section it feeds
    goes silently empty. Test for absence of a note instead."""
    if "note" not in frame.columns: return frame
    return frame[frame["note"].fillna("").astype(str).str.strip() == ""]

def seed_of(path):
    """Seed as an integer. Lexicographic ordering would put seed10 before seed2 and pick a
    different ten models than the manuscript's for the designated control."""
    m = re.findall(r"seed(\d+)", os.path.basename(path))
    assert m, f"cannot read a seed from {path}"
    return int(m[-1])

def holm(p):
    p = np.asarray(p, dtype=float); o = np.argsort(p); k = len(p); run = 0.0
    adj = np.empty(k)
    for r_, i_ in enumerate(o):
        run = max(run, (k-r_)*p[i_]); adj[i_] = min(run, 1.0)
    return adj

def boot_p(d):
    n_le = int((d <= 0).sum()); n_ge = int((d >= 0).sum()); B = len(d)
    return float(min(1.0, 2*min((1+n_le)/(1+B), (1+n_ge)/(1+B))))

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in ["train.csv","valid.csv","test.csv"]],
               ignore_index=True)[["Drug_ID","smiles","label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)
assert len(df) == 7255, f"{len(df)} compounds, expected 7255"
SCAF = {s: safe_scaffold(s) for s in df.canonical}

TOX = {"aromatic_nitro":"[a][N+](=O)[O-]","aromatic_amine":"[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
 "aromatic_nitroso":"[a][NX2]=[OX1]","alkyl_nitrite":"[CX4][OX2][NX2]=[OX1]",
 "nitrosamine":"[NX3][NX2]=[OX1]","epoxide":"C1OC1","aziridine":"C1NC1",
 "azide":"[NX2]=[NX2+]=[NX1-]","azo_aromatic":"[a][NX2]=[NX2][a]",
 "aliphatic_halide":"[CX4][F,Cl,Br,I]","hetero_hetero_bond":"[N,O;H1,H2][N,O;H1,H2]",
 "polycyclic_aromatic":"c1ccc2ccccc2c1","aromatic_hydroxylamine":"[a][NX3;H1][OX2;H1]",
 "michael_acceptor":"[CX3]=[CX3][CX3]=[OX1]","aromatic_azoxy":"[a][NX2]=[NX3+]([O-])[a]",
 "nitrogen_mustard":"[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
 "aromatic_hydrazine":"[a][NX3;H1][NX3;H2]","alkyl_sulfonate":"[CX4][OX2][SX4](=[OX1])(=[OX1])"}
PAT = {k: Chem.MolFromSmarts(v) for k, v in TOX.items()}
HETERO = [k for k in TOX if k != "polycyclic_aromatic"]

def eval_set(frame, labels={1}, alerts=None):
    pats = PAT if alerts is None else {k: PAT[k] for k in alerts}
    ev = []
    for smi, lab in zip(frame["canonical"], frame["label"]):
        if lab not in labels: continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None or mol.GetNumAtoms() == 0: continue
        t = set()
        for p in pats.values():
            for mt in mol.GetSubstructMatches(p): t.update(mt)
        if not t or len(t) == mol.GetNumAtoms(): continue
        ev.append({"smiles": smi, "mol": mol, "tox": t, "n_atoms": mol.GetNumAtoms()})
    return ev, np.array([len(e["tox"])/e["n_atoms"] for e in ev])

def per_molecule_auroc(ev, scores):
    out = np.full(len(ev), np.nan)
    for i, (e, sc) in enumerate(zip(ev, scores)):
        if sc is None or len(sc) != e["n_atoms"]: continue
        y = np.zeros(e["n_atoms"]); y[list(e["tox"])] = 1
        if 0 < y.sum() < len(y): out[i] = _auc(y, np.asarray(sc, dtype=float))
    return out

def common_support(per, ref):
    ok = ~np.isnan(ref) & ~np.isnan(per).any(axis=0)
    assert ok.sum() >= 0.90*len(ref), f"only {int(ok.sum())} of {len(ref)} scored on both"
    return per[:, ok], ref[ok], ok

def paired_draws(per, ref, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); n_mod, n_mol = per.shape
    out = np.empty(n_boot)
    for b in range(n_boot):
        mi = rng.integers(0, n_mol, n_mol); si = rng.integers(0, n_mod, n_mod)
        out[b] = per[np.ix_(si, mi)].mean() - ref[mi].mean()
    return out

def tie_fraction(ev, scores, k_frac=K_FRAC, arbitrary_only=False):
    """Share of the retained set that a tie at the cut decides.

    Two definitions. The default is the manuscript's: everything from the tied block that
    reaches the retained set counts, so a continuous ranking still returns 1/k for the atom
    at the cut itself. With arbitrary_only, a tied block that fits the retained set exactly
    counts as nothing, because then the tie decides no ordering that matters -- only a block
    larger than the space left is genuinely arbitrary. The second is the stricter reading and
    the one that supports a claim about arbitrariness."""
    out = []
    for e, sc in zip(ev, scores):
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        v = np.sort(np.asarray(sc, dtype=float))[::-1]; cut = v[k-1]
        above = int((v > cut).sum()); at = int((v == cut).sum())
        room = k - above
        out.append(0.0 if (arbitrary_only and at <= room) else room/k)
    return float(np.mean(out))

prim = pd.read_csv(f"{R}/primary_test.csv")
# ordered by seed rather than by filename. Every one of the twenty is used and the summaries
# over them are symmetric, so the order does not change a result; it makes the file a model
# maps to readable, and it is the ordering any later subset would have to take.
FILES = {"gin": sorted([f for f in glob.glob(f"{EXPL}/scores_ig_seed*.npz")
                        if "_gat_" not in f and "_gcn_" not in f and "no_degree" not in f],
                       key=seed_of),
         "gat": sorted(glob.glob(f"{EXPL}/scores_ig_gat_seed*.npz"), key=seed_of),
         "gcn": sorted(glob.glob(f"{EXPL}/scores_ig_gcn_seed*.npz"), key=seed_of)}
for a, v in FILES.items(): assert len(v) == 20, f"{a}: {len(v)} files, expected 20"
print(f"designated test set {len(prim)}, attribution scores for three architectures")

designated test set 726, attribution scores for three architectures


## 2. The regime decomposition on the threshold-free measure

The paper reports $+0.442$ and $-0.231$ for the two regimes on the lift scale, and calls the
threshold-free measure primary. The two statements do not sit together. Alert membership is
the same target under either measure and the stored scores are the same, so the decomposition
can be recomputed on the scale the paper prefers.

The evaluation set of a regime is built from that regime's alerts alone, so each has its own
molecules, its own chance level and its own tie behaviour. The tie fraction is reported per
regime: if the two differ, the lift measure was not doing the same work in each, which is
worth knowing whichever way the comparison falls.

In [5]:
PUBLISHED_FULL = {"atom degree": 0.580, "trained GIN": 0.637,
                  "trained GAT": 0.595, "trained GCN": 0.598}

def load_scores(path, ev, smi2i):
    z = np.load(path)
    need = {f"m{smi2i[e['smiles']]}" for e in ev}
    assert need <= set(z.files), (
        f"{os.path.basename(path)} is missing {len(need - set(z.files))} of the keys this "
        "evaluation set needs; the stored index does not match the set rebuilt here")
    return [z[f"m{smi2i[e['smiles']]}"] for e in ev]

def build_regime_auroc():
    """Both regimes, three architectures, on the measure the paper calls primary."""
    base_ev, _ = eval_set(prim)                       # the designated evaluation set
    smi2i = {e["smiles"]: i for i, e in enumerate(base_ev)}
    rows = []
    # the full library is not a new result; it is the positive control. Every function in
    # this notebook is a rewrite, and if any of them differs from the one that produced the
    # published numbers, all three sections are wrong together and nothing else would say so.
    for reg, alerts in [("full", None), ("no_polycyclic", HETERO),
                        ("polycyclic_only", ["polycyclic_aromatic"])]:
        ev, ch = eval_set(prim, alerts=alerts)
        ev = [e for e in ev if e["smiles"] in smi2i]   # scores exist only for the base set
        assert len(ev) >= 20, f"{reg}: only {len(ev)} molecules with stored scores"
        dg = [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float) for e in ev]
        d_auc = per_molecule_auroc(ev, dg)
        ok_d = ~np.isnan(d_auc)
        rows.append({"regime": reg, "arm": "atom degree", "n_eval": int(ok_d.sum()),
                     "n_models": 1,
                     "auroc": float(np.nanmean(d_auc)),
                     "difference": np.nan, "ci_low": np.nan, "ci_high": np.nan,
                     "p_raw": np.nan, "tie_fraction": tie_fraction(ev, dg),
                     "tie_arbitrary": tie_fraction(ev, dg, arbitrary_only=True)})
        if reg == "full":
            got = float(np.nanmean(d_auc))
            assert abs(got - PUBLISHED_FULL["atom degree"]) < 0.005, (
                f"degree AUROC on the full library is {got:.4f}, published 0.580; the "
                "rewritten scoring path does not reproduce the manuscript")
        for arch in ["gin", "gat", "gcn"]:
            sc_all = [load_scores(f, ev, smi2i) for f in FILES[arch]]
            per = np.array([per_molecule_auroc(ev, s_) for s_ in sc_all])
            p_, r_, ok = common_support(per, d_auc)
            dr = paired_draws(p_, r_, seed=SEED+7)
            lo, hi = np.percentile(dr, [2.5, 97.5])
            tf = float(np.mean([tie_fraction(ev, s_) for s_ in sc_all]))
            ta = float(np.mean([tie_fraction(ev, s_, arbitrary_only=True) for s_ in sc_all]))
            rows.append({"regime": reg, "arm": f"trained {arch.upper()}",
                         "n_eval": int(ok.sum()), "n_models": len(per),
                         "auroc": float(p_.mean()),
                         "difference": float(p_.mean()-r_.mean()),
                         "ci_low": lo, "ci_high": hi, "p_raw": boot_p(dr),
                         "tie_fraction": tf, "tie_arbitrary": ta})
            if reg == "full":
                pub = PUBLISHED_FULL[f"trained {arch.upper()}"]
                assert abs(p_.mean() - pub) < 0.005, (
                    f"{arch} AUROC on the full library is {p_.mean():.4f}, published {pub}")
            print(f"    {reg:16s} {arch}: auroc {p_.mean():.4f}  "
                  f"diff {p_.mean()-r_.mean():+.4f} [{lo:+.4f}, {hi:+.4f}]  "
                  f"p {boot_p(dr):.4f}")
    out = pd.DataFrame(rows)
    ok = out.p_raw.notna()
    # Table S15 assigns Holm to the regime decomposition across the six cells of the two
    # reduced regimes; the full-library row is a control and stays outside that family
    fam = ok & (out.regime != "full")
    out.loc[fam, "p_holm"] = holm(out.loc[fam, "p_raw"].values)
    out["resolved"] = out.p_holm < 0.05
    return out

reg_auc = cached("table97_regime_auroc.csv", build_regime_auroc)
print()
print(reg_auc.round(4).to_string(index=False))
print()
print("  On the lift scale the manuscript reports +0.442 and -0.231 for GIN.")
print("  Tie fractions per regime are in the last column; on the full library the")
print("  degree ranking carries 0.87 and the attributions about 0.19.")

    full             gin: auroc 0.6368  diff +0.0564 [+0.0187, +0.0926]  p 0.0036
    full             gat: auroc 0.5952  diff +0.0148 [-0.0068, +0.0365]  p 0.1680
    full             gcn: auroc 0.5982  diff +0.0179 [-0.0127, +0.0482]  p 0.2538
    no_polycyclic    gin: auroc 0.7150  diff +0.1925 [+0.1508, +0.2317]  p 0.0002
    no_polycyclic    gat: auroc 0.5627  diff +0.0402 [+0.0033, +0.0771]  p 0.0338
    no_polycyclic    gcn: auroc 0.5022  diff -0.0203 [-0.0663, +0.0264]  p 0.3934
    polycyclic_only  gin: auroc 0.4950  diff -0.1723 [-0.2345, -0.1095]  p 0.0002
    polycyclic_only  gat: auroc 0.6279  diff -0.0395 [-0.0999, +0.0179]  p 0.1910
    polycyclic_only  gcn: auroc 0.7116  diff +0.0443 [-0.0084, +0.0996]  p 0.1000
  table97_regime_auroc.csv: written

         regime         arm  n_eval  n_models  auroc  difference  ci_low  ci_high  p_raw  tie_fraction  tie_arbitrary  p_holm  resolved
           full atom degree     237         1 0.5803         NaN     NaN      NaN    NaN 

## 3. The structural ranking panel on the corrected partitions

Atom degree was chosen as the baseline on a screen of eleven rankings across three regimes,
run on the designated partition. Every comparison in the paper then used it as the reference,
on the same molecules. The screen was never repeated, and the paper reports that the
baseline's own agreement moves between $0.536$ and $0.607$ across the corrected partitions,
which is a wider range than any effect it reports.

None of the eleven rankings needs a model, so the screen transports at no cost.

In [6]:
def scaffold_split_random_order(d, seed, frac=(0.8, 0.1, 0.1)):
    """The corrected splitter of Part 17, with the same two groups pinned to training."""
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values): groups[SCAF[smi]].append(i)
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    budget = min(b-a, n-b)
    over = [g for g in groups.values() if len(g) > budget]
    rest = [g for g in groups.values() if len(g) <= budget]
    rng = np.random.default_rng(seed); rng.shuffle(rest)
    tr = [i for g in over for i in g]; va, te = [], []
    for g in rest:
        if len(tr)+len(g) <= a: tr += g
        elif len(tr)+len(va)+len(g) <= b: va += g
        else: te += g
    return d.loc[tr], d.loc[va], d.loc[te]

def rankings_for(mol):
    """The eleven structural rankings of the panel, as defined in the manuscript."""
    A = list(mol.GetAtoms()); n = len(A)
    deg = np.array([a.GetDegree() for a in A], dtype=float)
    het = np.array([0.0 if a.GetSymbol() == "C" else 1.0 for a in A])
    arom = np.array([float(a.GetIsAromatic()) for a in A])
    ring = np.array([float(a.IsInRing()) for a in A])
    hyd = np.array([float(a.GetTotalNumHs()) for a in A])
    chg = np.array([abs(a.GetFormalCharge()) for a in A], dtype=float)
    nbr_het = np.array([sum(0.0 if b.GetSymbol() == "C" else 1.0
                            for b in a.GetNeighbors()) for a in A])
    # the manuscript defines rarity over element, degree and aromaticity together; counting
    # the element alone would be a different panel from the one that chose the baseline
    key = [(a.GetSymbol(), a.GetDegree(), a.GetIsAromatic()) for a in A]
    cnt = Counter(key)
    rare = np.array([-cnt[k] for k in key], dtype=float)
    return {"degree": deg, "heteroatom": het, "aromatic": arom, "in_ring": ring,
            "hydrogens": hyd, "abs_formal_charge": chg, "hetero_neighbours": nbr_het,
            "hetero_or_neighbour": np.maximum(het, np.minimum(nbr_het, 1.0)),
            "hetero_plus_degree": het + deg/max(deg.max(), 1),
            "degree_deviation": np.abs(deg - deg.mean()), "rarity": rare}

def mean_overlap(ev, scores, k_frac=K_FRAC, n_tie=N_TIE, base=0):
    """Share of the retained set inside an alert, averaged over tie-breaks, per molecule.

    Ties are broken by a random secondary key under lexsort, seeded per molecule so that
    every ranking sees the same keys for the same molecule."""
    out = np.zeros(len(ev))
    for i, (e, sc) in enumerate(zip(ev, scores)):
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        v = -np.asarray(sc, dtype=float)
        acc = 0.0
        for j in range(n_tie):
            keys = np.random.default_rng([base, j, i]).random(n)
            order = np.lexsort((keys, v))
            acc += len(set(order[:k].tolist()) & e["tox"])/k
        out[i] = acc/n_tie
    return out

def panel_on(frame, seed_tag):
    out = []
    for reg, alerts in [("full", None), ("no_polycyclic", HETERO),
                        ("polycyclic_only", ["polycyclic_aromatic"])]:
        ev, ch = eval_set(frame, alerts=alerts)
        if len(ev) < 20:
            out.append({"partition": seed_tag, "regime": reg, "ranking": "--",
                        "note": f"only {len(ev)} molecules"}); continue
        R_ = {k: [] for k in rankings_for(ev[0]["mol"])}
        for e in ev:
            for k, v in rankings_for(e["mol"]).items(): R_[k].append(v)
        for k, sc in R_.items():
            ov = mean_overlap(ev, sc, base=seed_tag)
            rng = np.random.default_rng(SEED+seed_tag)
            bb = np.empty(N_BOOT)          # the decision is lo > 1.0, and Table 8 has
            for b in range(N_BOOT):        # cells whose lower bound sits at 1.00
                mi = rng.integers(0, len(ev), len(ev))
                bb[b] = ov[mi].mean()/ch[mi].mean()
            lo, hi = np.percentile(bb, [2.5, 97.5])
            out.append({"partition": seed_tag, "regime": reg, "ranking": k,
                        "n_eval": len(ev), "lift": float(ov.mean()/ch.mean()),
                        "ci_low": lo, "ci_high": hi,
                        "above_chance": bool(lo > 1.0), "note": ""})
    return pd.DataFrame(out)

PUBLISHED_PANEL = {"full": 1.214, "no_polycyclic": 1.278, "polycyclic_only": 1.162}
EXPECTED_EVAL = {201: 302, 202: 349, 203: 257}

def build_panel():
    parts = []
    # the designated partition first, as a control on the rewritten ranking definitions
    p0 = panel_on(prim, 0)
    for reg, want in PUBLISHED_PANEL.items():
        got = p0[(p0.ranking == "degree") & (p0.regime == reg)]
        assert len(got) == 1 and abs(float(got.lift.iloc[0]) - want) < 0.01, (
            f"degree lift on the designated partition, {reg}: "
            f"{float(got.lift.iloc[0]):.4f} against a published {want}; the rankings "
            "rebuilt here are not the ones that chose the baseline")
    parts.append(p0)
    print("    designated partition reproduces the published panel")
    for sd in PART_SEEDS:
        _, _, te = [x.reset_index(drop=True) for x in scaffold_split_random_order(df, sd)]
        # the split is rebuilt rather than read, so it is anchored to a published count
        ev, _ = eval_set(te)
        assert len(ev) == EXPECTED_EVAL[sd], (
            f"partition {sd}: {len(ev)} mutagenic evaluation molecules, expected "
            f"{EXPECTED_EVAL[sd]}; the reconstructed split does not match Part 17")
        parts.append(panel_on(te, sd))
        print(f"    partition {sd} done ({len(ev)} evaluation molecules)")
    return pd.concat(parts, ignore_index=True)

panel = cached("table98_panel_by_partition.csv", build_panel)
print()
w = rows_ok(panel)
piv = w.pivot_table(index=["partition","ranking"], columns="regime", values="above_chance")
# a regime that could not be computed is missing, not passed; .all() would skip the NaN
allthree = piv.notna().all(axis=1) & piv.fillna(False).astype(bool).all(axis=1)
print("rankings above chance in every regime, by partition:")
for sd in [0] + PART_SEEDS:                 # 0 is the designated partition, the control
    if sd not in allthree.index.get_level_values(0):
        print(f"  {sd}: not computed"); continue
    a = allthree.loc[sd]
    print(f"  {'designated' if sd == 0 else sd}: {sorted(a[a].index)}")
print()
print("  On the designated partition the manuscript reports atom degree as the only one.")

    designated partition reproduces the published panel
    partition 201 done (302 evaluation molecules)
    partition 202 done (349 evaluation molecules)
    partition 203 done (257 evaluation molecules)
  table98_panel_by_partition.csv: written

rankings above chance in every regime, by partition:
  designated: ['degree']
  201: ['degree']
  202: ['degree']
  203: ['degree']

  On the designated partition the manuscript reports atom degree as the only one.


## 4. The specificity check on the corrected partitions

The claim that a trained GIN ranks alert atoms relatively lower where they do not carry the
endpoint reaches the abstract, and it comes from the partition drawn with the splitter this
paper discredits. The models for the corrected partitions were fitted in Part 17 and are on
disk; only the signed attributions have to be recomputed, on each partition's own mutagenic
and non-mutagenic evaluation sets.

Five models per partition against ten on the designated one, so the intervals will be wider
here whatever the effect is.

In [7]:
HP_GIN = HP_ALL["gin"]

def signed_ig(model, graphs, n_steps=32, bs=64):
    was, saved = model.training, [p.requires_grad for p in model.parameters()]
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    out = []
    for batch in DataLoader(graphs, batch_size=bs, shuffle=False):
        batch = batch.to(DEVICE); x0 = batch.x.clone()
        total = torch.zeros_like(x0)
        for al in torch.linspace(1.0/n_steps, 1.0, n_steps):
            xa = (x0*al).detach().requires_grad_(True)
            lg = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
            total += torch.autograd.grad(lg.sum(), xa)[0]
        attr = (x0*(total/n_steps)).sum(1).detach().cpu().numpy()
        b = batch.batch.cpu().numpy()
        for gi in range(batch.num_graphs): out.append(attr[b == gi])
    for p, f in zip(model.parameters(), saved): p.requires_grad_(f)
    model.train(was)
    return out

EXPECTED_EVAL_DOD = {0: 237, 201: 302, 202: 349, 203: 257}
EXPECTED_NEG_DOD  = {0: 168}     # the manuscript gives this count only for the designated one
# designated partition, ten models: the difference of differences and both of its components
PUBLISHED_DOD, PUBLISHED_DOD_MUT, PUBLISHED_DOD_NEG = -0.0498, 0.0890, 0.0393
DOD_TOL = 0.005                  # fixed in the pre-registration, not to be widened here

def one_partition(sd, frame, weights):
    """The difference of differences on one partition, with an interval on each component.

    Both components are collected, not just their difference. A negative difference of
    differences only means what the paper says it means if the mutagenic component is
    positive: a model that sits below the baseline on both sets, further below on the
    non-mutagenic one, produces the same negative number for the opposite reason."""
    pos, _ = eval_set(frame, {1}); neg, _ = eval_set(frame, {0})
    assert len(pos) == EXPECTED_EVAL_DOD[sd], (
        f"partition {sd}: {len(pos)} mutagenic evaluation molecules, expected "
        f"{EXPECTED_EVAL_DOD[sd]}; the reconstructed split does not match")
    if sd in EXPECTED_NEG_DOD:
        assert len(neg) == EXPECTED_NEG_DOD[sd], (
            f"partition {sd}: {len(neg)} non-mutagenic evaluation molecules, expected "
            f"{EXPECTED_NEG_DOD[sd]}")
    pg = [smiles_to_data(e["smiles"], 1.0) for e in pos]
    ng = [smiles_to_data(e["smiles"], 0.0) for e in neg]
    d_pos = per_molecule_auroc(pos, [np.array([a.GetDegree() for a in e["mol"].GetAtoms()],
                                              dtype=float) for e in pos])
    d_neg = per_molecule_auroc(neg, [np.array([a.GetDegree() for a in e["mol"].GetAtoms()],
                                              dtype=float) for e in neg])
    P, N = [], []
    for w in weights:
        mdl = build_gnn("gin", HP_GIN["hidden"], HP_GIN["n_layers"], HP_GIN["dropout"],
                        HP_GIN.get("residual", True))
        mdl.load_state_dict(torch.load(w, map_location=DEVICE,
                                       weights_only=False)["state_dict"]); mdl.eval()
        P.append(per_molecule_auroc(pos, signed_ig(mdl, pg)))
        N.append(per_molecule_auroc(neg, signed_ig(mdl, ng)))
        del mdl
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    P, p_ref, ok_p = common_support(np.array(P), d_pos)
    N, n_ref, ok_n = common_support(np.array(N), d_neg)
    rng = np.random.default_rng(SEED + sd)
    bb_p = np.empty(N_BOOT); bb_n = np.empty(N_BOOT); bb = np.empty(N_BOOT)
    for b in range(N_BOOT):
        si = rng.integers(0, P.shape[0], P.shape[0])
        pi = rng.integers(0, P.shape[1], P.shape[1])
        ni = rng.integers(0, N.shape[1], N.shape[1])
        bb_p[b] = P[np.ix_(si, pi)].mean() - p_ref[pi].mean()
        bb_n[b] = N[np.ix_(si, ni)].mean() - n_ref[ni].mean()
        bb[b] = bb_n[b] - bb_p[b]
    dp0 = float(P.mean()-p_ref.mean()); dn0 = float(N.mean()-n_ref.mean())
    lo, hi = np.percentile(bb, [2.5, 97.5])
    plo, phi = np.percentile(bb_p, [2.5, 97.5])
    nlo, nhi = np.percentile(bb_n, [2.5, 97.5])
    return {"partition": sd, "n_models": len(weights),
            "n_mutagenic": int(ok_p.sum()), "n_non_mutagenic": int(ok_n.sum()),
            "diff_mutagenic": dp0, "mut_lo": plo, "mut_hi": phi,
            "diff_non_mutagenic": dn0, "neg_lo": nlo, "neg_hi": nhi,
            "dod": dn0-dp0, "ci_low": lo, "ci_high": hi,
            "p_raw": boot_p(bb), "note": ""}

def build_dod_by_partition():
    rows = []
    # the designated partition first: signed_ig is rewritten here and must reproduce -0.050
    found = sorted(glob.glob(f"{MODELS}/gin_attr_seed*.pt"), key=seed_of)
    assert len(found) >= 10, f"designated partition: {len(found)} weights, expected at least 10"
    ws0 = found[:10]
    print(f"    designated control uses seeds {[seed_of(w) for w in ws0]}")
    r0 = one_partition(0, prim, ws0)
    # both components as well as their difference: a matching difference of differences with
    # mismatched components would mean the two sets had been swapped or rescored
    for name, got, want in [("dod", r0["dod"], PUBLISHED_DOD),
                            ("mutagenic component", r0["diff_mutagenic"], PUBLISHED_DOD_MUT),
                            ("non-mutagenic component", r0["diff_non_mutagenic"], PUBLISHED_DOD_NEG)]:
        assert abs(got - want) < DOD_TOL, (
            f"designated partition, {name}: {got:+.4f} against a published {want:+.4f}; the "
            "signed attribution path rewritten here does not reproduce the manuscript")
    rows.append(r0)
    print(f"    designated: dod {r0['dod']:+.4f}, components {r0['diff_mutagenic']:+.4f} / "
          f"{r0['diff_non_mutagenic']:+.4f}, all three reproduce the published values")
    for sd in PART_SEEDS:
        _, _, te = [x.reset_index(drop=True) for x in scaffold_split_random_order(df, sd)]
        # only the counts are needed here; one_partition rebuilds everything it uses
        pos, _ = eval_set(te, {1}); neg, _ = eval_set(te, {0})
        if len(pos) < 20 or len(neg) < 20:
            rows.append({"partition": sd, "note": f"pos {len(pos)} neg {len(neg)}, too few"})
            print(f"    {sd}: pos {len(pos)} neg {len(neg)}, skipped"); continue
        ws = sorted(glob.glob(f"{MODELS}/gin_part{sd}_seed*.pt"), key=seed_of)
        assert len(ws) == 5, (f"partition {sd}: {len(ws)} stored weights, expected 5; "
                              "the pre-registration fixes five models per partition")
        rows.append(one_partition(sd, te, ws))
        print(f"    {sd}: dod {rows[-1]['dod']:+.4f} "
              f"[{rows[-1]['ci_low']:+.4f}, {rows[-1]['ci_high']:+.4f}]  "
              f"p {rows[-1]['p_raw']:.4f}")

    out = pd.DataFrame(rows)
    # Table S15 assigns no correction to the replication family: three pre-specified
    # partitions, decision on the interval. Applying Holm here would contradict the paper's
    # own decision rule, which is the thing this paper asks others to write down.
    out["resolved"] = (out.ci_high < 0) | (out.ci_low > 0)
    return out

dod_rep = cached("table99_dod_by_partition.csv", build_dod_by_partition)
print()
print(dod_rep.round(4).to_string(index=False))
print()
print("  Designated partition, ten models: dod -0.050 (-0.090 to -0.011); the manuscript")
print("  reports Holm p = 0.037 for it across the three architectures. The replication family")
print("  carries no correction, as Table S15 specifies, and is read on the interval.")
print("  The pre-registered direction is negative, and the reading also requires the")
print("  mutagenic component to be positive on the same partition.")

    designated control uses seeds [20000, 20001, 20002, 20003, 20004, 20005, 20006, 20007, 20008, 20009]
    designated: dod -0.0498, components +0.0890 / +0.0393, all three reproduce the published values
    201: dod -0.0887 [-0.1517, -0.0306]  p 0.0026
    202: dod +0.0105 [-0.0561, +0.0700]  p 0.7157
    203: dod -0.0137 [-0.0668, +0.0392]  p 0.6189
  table99_dod_by_partition.csv: written

 partition  n_models  n_mutagenic  n_non_mutagenic  diff_mutagenic  mut_lo  mut_hi  diff_non_mutagenic  neg_lo  neg_hi     dod  ci_low  ci_high  p_raw note  resolved
         0        10          237              168          0.0890  0.0478  0.1254              0.0393 -0.0087  0.0827 -0.0498 -0.0907  -0.0114 0.0110           True
       201         5          302              142          0.0561 -0.0104  0.1230             -0.0326 -0.1037  0.0351 -0.0887 -0.1517  -0.0306 0.0026           True
       202         5          349              104          0.1160  0.0684  0.1656              0.1265  0.

## 5. What the non-nested search actually cost

The hyperparameter search was run once on the designated partition and the selected
configurations were reused in the cross-validation, so the 725 compounds of that partition's
validation set appear in the test folds. The manuscript states this and adds that the
comparison between models is presumably less affected, since every model was tuned on the same
compounds under the same budget, but that it was not tested. Presumably is not a measurement.

It can be measured without fitting anything. The out-of-fold predictions are stored for every
compound, so they can be split into the 725 that the search saw and the 6,530 it did not, and
each model scored on both. Absolute optimism is expected on the exposed set and is not the
question; the question is whether it is of similar size for every model, because the family
contrast is a difference between models and only a difference in optimism can move it.

The two subsets are not matched on difficulty, so the absolute gap between them means little.
Only the spread of that gap across models is interpretable, and that is what is reported,
with an interval on the difference between the two family means taken from the same
resampling that every model sees.

In [8]:
def build_tuning_optimism():
    """Optimism on the compounds the hyperparameter search saw, model by model.

    The row order of the stored predictions is the curated frame, which this notebook rebuilds
    the same way; the label vector is checked against it before anything is computed."""
    p = f"{R}/oof_predictions.npz"
    if not os.path.exists(p):
        return pd.DataFrame([{"note": "oof_predictions.npz not found"}])
    z = np.load(p, allow_pickle=True)
    y = np.asarray(z["y"])
    assert len(y) == len(df), f"stored predictions cover {len(y)} compounds, curated frame {len(df)}"
    assert np.array_equal(df.label.values.astype(float), y), (
        "the stored predictions are not in the order of the curated frame; the split below "
        "would be applied to the wrong compounds")

    valid = pd.read_csv(f"{R}/primary_valid.csv")
    tuned = df.canonical.isin(set(valid.canonical)).values
    assert tuned.sum() == len(valid), (
        f"{int(tuned.sum())} of {len(valid)} tuning compounds matched in the curated frame")
    print(f"  {int(tuned.sum())} compounds were seen by the search, {int((~tuned).sum())} were not")

    FAMILY = {"XGBoost": "classical", "RandomForest": "classical",
              "XGBoost_RDKit2D": "classical", "GCN": "graph", "GIN": "graph", "GAT": "graph"}
    # out-of-fold ROC-AUC as published in Table 1. A key that holds a different model's
    # predictions would give plausible optimism numbers and nothing else would say so.
    PUBLISHED_OOF = {"XGBoost": 0.832, "RandomForest": 0.821, "XGBoost_RDKit2D": 0.845,
                     "GCN": 0.843, "GIN": 0.845, "GAT": 0.854}
    missing = [m_ for m_ in FAMILY if m_ not in z.files]
    assert not missing, (f"stored predictions are missing {missing}; the family means below "
                         "would be taken over an incomplete set")

    it = np.where(tuned)[0]; ir = np.where(~tuned)[0]
    rows, draws = [], {}
    for m_, fam in FAMILY.items():
        pr = np.asarray(z[m_])
        overall = _auc(y, pr)
        assert abs(overall - PUBLISHED_OOF[m_]) < 0.002, (
            f"{m_}: out-of-fold ROC-AUC over all compounds is {overall:.4f} against a "
            f"published {PUBLISHED_OOF[m_]}; this key does not hold that model")
        a_t = _auc(y[tuned], pr[tuned]); a_r = _auc(y[~tuned], pr[~tuned])
        # one interval per model, and the same resampling for every model so that the
        # difference between two of them can be read off the same draws
        rng = np.random.default_rng(SEED)
        bb = np.empty(N_BOOT)
        for b in range(N_BOOT):
            st = rng.choice(it, len(it)); sr = rng.choice(ir, len(ir))
            bb[b] = _auc(y[st], pr[st]) - _auc(y[sr], pr[sr])
        draws[m_] = bb
        lo, hi = np.percentile(bb, [2.5, 97.5])
        rows.append({"model": m_, "family": fam, "auc_all": overall,
                     "auc_tuned": a_t, "auc_rest": a_r,
                     "optimism": a_t - a_r, "ci_low": lo, "ci_high": hi, "note": ""})
        print(f"    {m_:18s} {fam:9s} all {overall:.4f}  tuned {a_t:.4f}  rest {a_r:.4f}  "
              f"optimism {a_t-a_r:+.4f} [{lo:+.4f}, {hi:+.4f}]")

    # the pre-registered quantity is the difference between the families, not each model's
    # own optimism, so it carries its own interval off the shared draws
    G = [m_ for m_, f_ in FAMILY.items() if f_ == "graph"]
    C = [m_ for m_, f_ in FAMILY.items() if f_ == "classical"]
    gap = np.mean([draws[m_] for m_ in G], axis=0) - np.mean([draws[m_] for m_ in C], axis=0)
    glo, ghi = np.percentile(gap, [2.5, 97.5])
    out = pd.DataFrame(rows)
    out.loc[out.index[0], "family_gap"] = float(gap.mean())
    out.loc[out.index[0], "family_gap_lo"] = float(glo)
    out.loc[out.index[0], "family_gap_hi"] = float(ghi)
    print(f"    graph minus classical optimism: {gap.mean():+.4f} "
          f"[{glo:+.4f}, {ghi:+.4f}], against a family contrast of +0.0146")
    return out

opt = cached("table101_tuning_optimism.csv", build_tuning_optimism)
print()
if "optimism" in opt.columns:
    print(opt[["model","family","auc_all","auc_tuned","auc_rest","optimism",
               "ci_low","ci_high"]].round(4).to_string(index=False))
    print()
    print(f"  graph minus classical: {opt.family_gap.dropna().iloc[0]:+.4f} "
          f"[{opt.family_gap_lo.dropna().iloc[0]:+.4f}, "
          f"{opt.family_gap_hi.dropna().iloc[0]:+.4f}]")
    print(f"  spread across the six models: {opt.optimism.min():+.4f} to "
          f"{opt.optimism.max():+.4f}")
    print("  the family contrast the manuscript reports is +0.0146; a difference in optimism")
    print("  of that order would account for it, a much smaller one would not")
else:
    print(opt.to_string(index=False))

  725 compounds were seen by the search, 6530 were not
    XGBoost            classical all 0.8324  tuned 0.8511  rest 0.8307  optimism +0.0204 [-0.0098, +0.0483]
    RandomForest       classical all 0.8209  tuned 0.8614  rest 0.8176  optimism +0.0438 [+0.0145, +0.0708]
    XGBoost_RDKit2D    classical all 0.8453  tuned 0.8620  rest 0.8439  optimism +0.0181 [-0.0103, +0.0455]
    GCN                graph     all 0.8430  tuned 0.8438  rest 0.8430  optimism +0.0008 [-0.0297, +0.0301]
    GIN                graph     all 0.8454  tuned 0.8586  rest 0.8439  optimism +0.0147 [-0.0155, +0.0430]
    GAT                graph     all 0.8541  tuned 0.8421  rest 0.8552  optimism -0.0132 [-0.0443, +0.0163]
    graph minus classical optimism: -0.0267 [-0.0425, -0.0115], against a family contrast of +0.0146
  table101_tuning_optimism.csv: written

          model    family  auc_all  auc_tuned  auc_rest  optimism  ci_low  ci_high
        XGBoost classical   0.8324     0.8511    0.8307    0.0204 -0.009

## 6. Summary

In [9]:
print("regime decomposition on the threshold-free measure")
print(reg_auc[["regime","arm","n_eval","auroc","difference","ci_low","ci_high",
               "p_holm","resolved","tie_fraction","tie_arbitrary"]]
      .round(4).to_string(index=False))
for reg in reg_auc.regime.unique():
    g = reg_auc[(reg_auc.regime == reg) & (reg_auc.arm == "trained GIN")]
    if len(g):
        r = g.iloc[0]
        tag = "control" if reg == "full" else ("resolved" if r.get("resolved") else
                                               "not resolved")
        print(f"  {reg}: GIN {r.difference:+.4f}, {tag}")

if len(rows_ok(panel)):
    w = rows_ok(panel)
    piv = w.pivot_table(index=["partition","ranking"], columns="regime",
                        values="above_chance")
    # a regime that could not be computed is missing, not passed; a bare .all() skips the NaN
    allthree = piv.notna().all(axis=1) & piv.fillna(False).astype(bool).all(axis=1)
    print("\nstructural panel: rankings above chance in every regime")
    for sd in [0] + PART_SEEDS:             # 0 is the designated partition, the control
        if sd not in allthree.index.get_level_values(0):
            print(f"  {sd}: not computed"); continue
        a = allthree.loc[sd]
        print(f"  {'designated' if sd == 0 else sd}: {sorted(a[a].index)}")

if "dod" in dod_rep.columns:
    print("\nspecificity by partition")
    print(dod_rep[["partition","n_models","diff_mutagenic","diff_non_mutagenic",
                   "dod","ci_low","ci_high","resolved"]].round(4).to_string(index=False))
    rep_only = dod_rep[dod_rep.partition.isin(PART_SEEDS) & dod_rep.dod.notna()]
    sg = set(np.sign(rep_only.dod))
    print(f"  signs on the three corrected partitions {list(np.sign(rep_only.dod).astype(int))},"
          f" {'consistent' if len(sg) == 1 else 'INCONSISTENT'}")
    # the pre-registered reading: a negative difference of differences means what the paper
    # says only where the mutagenic component is positive on the same partition
    bad = rep_only[(rep_only.dod < 0) & (rep_only.diff_mutagenic <= 0)]
    if len(bad):
        print(f"  negative with a non-positive mutagenic component on partitions "
              f"{list(bad.partition.astype(int))}: the number replicates, the reading does not")
    else:
        print("  every negative difference of differences carries a positive mutagenic "
              "component")

if "optimism" in opt.columns:
    print("\ntuning optimism")
    print(opt[["model","family","auc_all","optimism","ci_low","ci_high"]]
          .round(4).to_string(index=False))
    print(f"  graph minus classical: {opt.family_gap.dropna().iloc[0]:+.4f} "
          f"[{opt.family_gap_lo.dropna().iloc[0]:+.4f}, "
          f"{opt.family_gap_hi.dropna().iloc[0]:+.4f}], "
          "against a family contrast of +0.0146")

num = {}
for _, r in reg_auc.iterrows():
    k = f"{r.regime}_{r['arm']}".lower().replace(" ", "_")
    num[f"regauroc_{k}"] = float(r.auroc)
    if pd.notna(r.difference):
        num[f"regauroc_{k}_diff"] = float(r.difference)
        num[f"regauroc_{k}_lo"] = float(r.ci_low); num[f"regauroc_{k}_hi"] = float(r.ci_high)
    if pd.notna(r.get("p_holm")): num[f"regauroc_{k}_holm"] = float(r.p_holm)
    num[f"regtie_{k}"] = float(r.tie_fraction)
    num[f"regtiearb_{k}"] = float(r.tie_arbitrary)     # the stricter of the two definitions
if len(rows_ok(panel)):
    for _, r in rows_ok(panel).iterrows():
        tag = "designated" if int(r.partition) == 0 else int(r.partition)
        num[f"panel{tag}_{r.ranking}_{r.regime}_lift"] = float(r.lift)
for _, r in dod_rep.iterrows():
    if pd.notna(r.get("dod")):
        tag = "designated" if int(r.partition) == 0 else int(r.partition)
        num[f"dodrep{tag}"] = float(r.dod)
        num[f"dodrep{tag}_lo"] = float(r.ci_low); num[f"dodrep{tag}_hi"] = float(r.ci_high)
        num[f"dodrep{tag}_mut"] = float(r.diff_mutagenic)
        num[f"dodrep{tag}_neg"] = float(r.diff_non_mutagenic)
if "optimism" in opt.columns:
    for _, r in opt.iterrows():
        num[f"opt_{r.model}"] = float(r.optimism)
        num[f"opt_{r.model}_lo"] = float(r.ci_low); num[f"opt_{r.model}_hi"] = float(r.ci_high)
    num["opt_family_gap"] = float(opt.family_gap.dropna().iloc[0])
    num["opt_family_gap_lo"] = float(opt.family_gap_lo.dropna().iloc[0])
    num["opt_family_gap_hi"] = float(opt.family_gap_hi.dropna().iloc[0])
pd.DataFrame(sorted(num.items()), columns=["quantity","value"]).to_csv(
    f"{R}/table100_part19_ledger.csv", index=False)
print(f"\n{len(num)} quantities written for merging into the manuscript ledger")
for f in ["table97_regime_auroc.csv","table98_panel_by_partition.csv",
          "table99_dod_by_partition.csv","table101_tuning_optimism.csv",
          "table100_part19_ledger.csv"]:
    print("  -", f, "yes" if os.path.exists(f"{R}/{f}") else "MISSING")
import shutil
shutil.make_archive("/content/part19_results", "zip", R)
try:
    from google.colab import files
    files.download("/content/part19_results.zip")
except ImportError:
    print("not in Colab; the archive is at /content/part19_results.zip")


regime decomposition on the threshold-free measure
         regime         arm  n_eval  auroc  difference  ci_low  ci_high  p_holm  resolved  tie_fraction  tie_arbitrary
           full atom degree     237 0.5803         NaN     NaN      NaN     NaN     False        0.8682         0.7364
           full trained GIN     237 0.6368      0.0564  0.0187   0.0926     NaN     False        0.1923         0.0187
           full trained GAT     237 0.5952      0.0148 -0.0068   0.0365     NaN     False        0.1885         0.0149
           full trained GCN     237 0.5982      0.0179 -0.0127   0.0482     NaN     False        0.1958         0.0241
  no_polycyclic atom degree     174 0.5225         NaN     NaN      NaN     NaN     False        0.8249         0.6454
  no_polycyclic trained GIN     174 0.7150      0.1925  0.1508   0.2317  0.0012      True        0.2041         0.0221
  no_polycyclic trained GAT     174 0.5627      0.0402  0.0033   0.0771  0.1352     False        0.1991         0.01

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>